# Notebook choropleth_map

This notebook generates an interactive choropleth map showing the percentage of burnt area for each province and territory for 2014 and 2023.

### Input data
- burnt_area.csv: Wildfire data including for each year and each province the area of the province (in hectare), the area burnt down by wildfire (in hectare) and the percentage of the area burnt down by wildfires
- canada_provinces.gpkg: Provinces and Territories of Canada data with crs EPSG:3347, but column "PRENAME" now as "province"

### Outputs
- choropleth_map.html: Interactive choropleth map for the years 2014 and 2023 showing how much percentage of each province and territory was burnt

### Key assumptions
- canada_provinces.gpkg is reprojected to EPSG:4326

In [1]:
import pandas as pd
import geopandas as gpd
import folium

In [2]:
# 1. Load Data
# Loading burnt_are data
burnt_area = pd.read_csv("../data/processed/choropleth_map/burnt_area.csv", sep = ",")
print("Data loaded successfully")

# Load canada data
canada = gpd.read_file("../data/processed/choropleth_map/canada_provinces.gpkg")
print("GPKG loaded successfully")

Data loaded successfully
GPKG loaded successfully


In [3]:
# 2. Preparations for interactive choropleth map
# Reproject Canadad data to EPSG:4326
TARGET_CRS = "EPSG:4326"
canada_map = canada.to_crs(TARGET_CRS).copy()

# Simplyfying th geometry --> necessary as this helps to shorten the loading time of the map
canada_map["geometry"] = canada_map["geometry"].simplify(
    tolerance=0.05,
    preserve_topology=False
)

# Merging canada geometry and burn_area together as a geodataframe
interactive_choropleth_gdf = canada_map[["province", "geometry"]].merge(
    burnt_area,
    on="province",
    how="left"
)

interactive_choropleth_gdf.head(5)

,province,geometry,year,size_ha,area_ha,burnt_percent
0,Newfoundland and Labrador,"MULTIPOLYGON (((-56.16833 50.86826, -56.15315 ...",2014.0,8832.0,3.973420e+07,0.022228
1,Newfoundland and Labrador,"MULTIPOLYGON (((-56.16833 50.86826, -56.15315 ...",2015.0,3520.0,3.973420e+07,0.008859
2,Newfoundland and Labrador,"MULTIPOLYGON (((-56.16833 50.86826, -56.15315 ...",2016.0,10316.0,3.973420e+07,0.025963
3,Newfoundland and Labrador,"MULTIPOLYGON (((-56.16833 50.86826, -56.15315 ...",2017.0,609.0,3.973420e+07,0.001533
4,Newfoundland and Labrador,"MULTIPOLYGON (((-56.16833 50.86826, -56.15315 ...",2020.0,3793.5,3.973420e+07,0.009547


## Interactive Choropleth map

The section 3. will generate one of the main outputs of this project an interactive choroplet map.
The map shows for each province and territory in Canada the percentage of area which got burn down by wildfires.
One layer is for the year 2014 and one layer is for the year 2023.
Tooltips provide informations on the province name and the exact percentage of area which burnt down.

In [4]:
# 3. Choropleth map showing for each province and terriory the percentage of burnt area
# Using the basemap Jawg.Sunny from Jawaglab
tile_url = "https://tile.jawg.io/jawg-sunny/{z}/{x}/{y}.png?access-token=tniPiJQZVj5mGbfqOx8aIkfbRAIb4oLMomGVDcYNw07EFi13vUfu7z5GRFLN6Al9"

attribution = ('<a href="https://jawg.io" title="Tiles Courtesy of Jawg Maps" target="_blank">&copy; <b>Jawg</b>Maps</a> &copy; <a href="https://www.openstreetmap.org/copyright">OpenStreetMap</a> contributors'

)

# Choose years
years = [2014, 2023]

# Choose bins
bins = [0, 1, 2, 3, 4, 5]

# Create Folium map object
choropleth = folium.Map(
    location=[62, -98],
    zoom_start=4,
    tiles=None
)

# Basemap with better name
folium.TileLayer(tiles=tile_url,
                 attr=attribution,
                 name="Basemap sunny"
).add_to(choropleth)

# Create layers
for i, year in enumerate(years):

    each_year = interactive_choropleth_gdf[interactive_choropleth_gdf["year"] == year]

    layer = folium.Choropleth(
        geo_data=each_year,
        name=f"Burned area [%] in {year}",
        data=each_year,  
        columns=["province", "burnt_percent"],
        key_on="feature.properties.province",
        fill_color="YlOrRd",
        fill_opacity=0.7,
        line_opacity=0.2,
        bins = bins,
        legend_name="Burned area [%]",
        show=True if i == 0 else False
    ).add_to(choropleth)

    # Remove colorbar so that only one is shown
    if i > 0:
        for key in list(layer._children):
            if key.startswith("color_map"):
                del layer._children[key]

    folium.GeoJson(
        each_year,
        style_function=lambda x: {
            "fillColor": "#ffffff00",
            "color": "#ffffff00",
            "weight": 0
        },
        tooltip=folium.GeoJsonTooltip(
            fields=["province", "burnt_percent"],
            aliases=["Province:", "Burned Area [%]:"],
            localize=True,
        )
    ).add_to(layer)

# Add Layer Control
folium.LayerControl().add_to(choropleth)

# Save interactive map
choropleth.save("../outputs/choropleth_map.html")

choropleth

**Interpretation:**

In the year 2014 all provinces and territories, expect for Northwest Territories, are in the yellow class, indicating that less than 1% of their area got burnt down by wildfires. In the year 2023 the provinces British Columbia, Alberta, Saskatchewan, Quebec and the territories Yukon and Northwest Territories are in orange and red classes. This indicates that in those provinces and territories more area got burnt down by wildfires in 2023 compared to 2014. A closer look at the map makes clear that also the other provinces Manitoba, Ontario, Newfoundland and Labrador have more area burnt down by wildfires 2023 than 2014, but as it still is under 1% of their whole area they stay in the same colour class as in 2014.